# Train Tiny10 CIFAR-10 bank

This notebook trains the Tiny10 CNN bank used for the layer-matching sanity check. It saves checkpoints in the format expected by the layer-matching analysis notebook.

In [ ]:
# @title 0. Mount Drive
from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
drive.mount("/content/drive", force_remount=False)


In [ ]:
# @title 1. Setup, imports, and locked config
!pip install -q flax optax tensorflow tensorflow-datasets pandas matplotlib tqdm

import os
import re
import json
import time
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import jax
import jax.numpy as jnp
import flax
import flax.linen as nn
from flax import serialization
from flax.training import train_state
from flax.core import freeze
import optax

import tensorflow as tf
import tensorflow_datasets as tfds

# Keep TensorFlow from taking GPU memory from JAX.
tf.config.experimental.set_visible_devices([], "GPU")

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())

# -----------------------------
# Bank / path settings
# -----------------------------
NUM_NETWORKS = 10
NUM_CLASSES = 10
DATASET_NAME = "cifar10"

# This flat path is chosen to match the layer-matching loader snippet:
#     MODEL_SAVE_PATH / f"tiny10_model_{i}.msgpack"
MODEL_SAVE_PATH = DRIVE_ROOT / "jax_models"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Training hyperparameters
# -----------------------------
BATCH_SIZE = 128
NUM_EPOCHS = 100
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.0

# Augmentation: standard lightweight CIFAR training augmentation.
USE_AUGMENTATION = True
PAD_PIXELS = 4

# Seeds used for the 10 independent models.
SEEDS = list(range(NUM_NETWORKS))

# Resume behavior: skip models whose final checkpoint already exists.
SKIP_IF_EXISTS = True

CONFIG = dict(
    num_networks=NUM_NETWORKS,
    dataset_name=DATASET_NAME,
    model_save_path=str(MODEL_SAVE_PATH),
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    label_smoothing=LABEL_SMOOTHING,
    use_augmentation=USE_AUGMENTATION,
    pad_pixels=PAD_PIXELS,
    seeds=SEEDS,
)

with open(MODEL_SAVE_PATH / "tiny10_training_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

CONFIG


In [ ]:
# @title 2. Tiny10 model definition: exact analysis-compatible architecture
class Tiny10(nn.Module):
    num_classes: int = 10

    @nn.compact
    def __call__(self, x, train: bool):
        activations = {}
        norm = partial(nn.BatchNorm, use_running_average=not train, momentum=0.9)

        x = nn.Conv(features=16, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv1_relu"] = x

        x = nn.Conv(features=16, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv2_relu"] = x

        x = nn.Conv(features=32, kernel_size=(3, 3), strides=2, padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv3_relu"] = x

        x = nn.Conv(features=32, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv4_relu"] = x

        x = nn.Conv(features=32, kernel_size=(3, 3), padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv5_relu"] = x

        x = nn.Conv(features=64, kernel_size=(3, 3), strides=2, padding=1)(x)
        x = norm()(x); x = nn.relu(x); activations["conv6_relu"] = x

        x = nn.Conv(features=64, kernel_size=(3, 3), padding=0, use_bias=False)(x)
        x = norm()(x); x = nn.relu(x); activations["conv7_relu"] = x

        x = nn.Conv(features=64, kernel_size=(1, 1), padding=0)(x)
        x = norm()(x); x = nn.relu(x); activations["conv8_relu"] = x

        x = jnp.mean(x, axis=(1, 2))
        x = nn.Dense(features=self.num_classes)(x)
        return x, activations


def natural_key(s):
    return [int(c) if c.isdigit() else c for c in re.split(r"(\d+)", s)]

key = jax.random.PRNGKey(0)
dummy_input = jnp.ones((1, 32, 32, 3), dtype=jnp.float32)
init_vars = Tiny10(num_classes=NUM_CLASSES).init(key, dummy_input, train=False)
_, dummy_acts = Tiny10(num_classes=NUM_CLASSES).apply(init_vars, dummy_input, train=False)
LAYER_NAMES = sorted(list(dummy_acts.keys()), key=natural_key)
print("Layers:", LAYER_NAMES)
print("Variable collections:", init_vars.keys())


In [ ]:
# @title 3. CIFAR-10 loading and preprocessing

def load_cifar10_numpy():
    # Load CIFAR-10 as full numpy arrays, normalized exactly as in the analysis snippet.
    train_ds = tfds.load(DATASET_NAME, split="train", batch_size=-1, as_supervised=True)
    test_ds = tfds.load(DATASET_NAME, split="test", batch_size=-1, as_supervised=True)

    x_train, y_train = tfds.as_numpy(train_ds)
    x_test, y_test = tfds.as_numpy(test_ds)

    x_train = (x_train.astype(np.float32) - 127.5) / 127.5
    x_test = (x_test.astype(np.float32) - 127.5) / 127.5
    y_train = y_train.astype(np.int32)
    y_test = y_test.astype(np.int32)
    return x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = load_cifar10_numpy()
print("train:", x_train.shape, y_train.shape, x_train.min(), x_train.max())
print("test: ", x_test.shape, y_test.shape, x_test.min(), x_test.max())


In [ ]:
# @title 4. Data augmentation and batching

def random_crop_flip_numpy(x, rng, pad=4):
    # Random crop + horizontal flip for NHWC CIFAR images in numpy.
    if pad <= 0:
        x_aug = x.copy()
    else:
        x_pad = np.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="reflect")
        n, h, w, c = x.shape
        ys = rng.integers(0, 2 * pad + 1, size=n)
        xs = rng.integers(0, 2 * pad + 1, size=n)
        x_aug = np.empty_like(x)
        for i, (yy, xx) in enumerate(zip(ys, xs)):
            x_aug[i] = x_pad[i, yy:yy + h, xx:xx + w, :]

    flips = rng.random(size=x_aug.shape[0]) < 0.5
    x_aug[flips] = x_aug[flips, :, ::-1, :]
    return x_aug


def iter_minibatches(x, y, batch_size, rng, train=True, augment=True):
    n = x.shape[0]
    indices = np.arange(n)
    if train:
        rng.shuffle(indices)
    for start in range(0, n, batch_size):
        idx = indices[start:start + batch_size]
        xb = x[idx]
        yb = y[idx]
        if train and augment:
            xb = random_crop_flip_numpy(xb, rng=rng, pad=PAD_PIXELS)
        yield xb.astype(np.float32), yb.astype(np.int32)


In [ ]:
# @title 5. TrainState, optimizer, train/eval steps
class TrainState(train_state.TrainState):
    batch_stats: flax.core.FrozenDict


def create_learning_rate_fn(num_train_examples):
    steps_per_epoch = int(np.ceil(num_train_examples / BATCH_SIZE))
    total_steps = steps_per_epoch * NUM_EPOCHS
    warmup_steps = steps_per_epoch * WARMUP_EPOCHS

    cosine_fn = optax.cosine_decay_schedule(
        init_value=BASE_LR,
        decay_steps=max(1, total_steps - warmup_steps),
        alpha=0.01,
    )

    if warmup_steps <= 0:
        return cosine_fn

    warmup_fn = optax.linear_schedule(
        init_value=0.0,
        end_value=BASE_LR,
        transition_steps=warmup_steps,
    )
    return optax.join_schedules([warmup_fn, cosine_fn], [warmup_steps])


def create_state(seed, num_train_examples):
    model = Tiny10(num_classes=NUM_CLASSES)
    rng = jax.random.PRNGKey(seed)
    variables = model.init(rng, jnp.ones((1, 32, 32, 3), dtype=jnp.float32), train=True)

    lr_fn = create_learning_rate_fn(num_train_examples)
    tx = optax.adamw(learning_rate=lr_fn, weight_decay=WEIGHT_DECAY)

    return TrainState.create(
        apply_fn=model.apply,
        params=variables["params"],
        tx=tx,
        batch_stats=variables["batch_stats"],
    )


def cross_entropy_loss(logits, labels):
    onehot = jax.nn.one_hot(labels, NUM_CLASSES)
    if LABEL_SMOOTHING > 0:
        onehot = optax.smooth_labels(onehot, LABEL_SMOOTHING)
    return optax.softmax_cross_entropy(logits, onehot).mean()


@jax.jit
def train_step(state, images, labels):
    def loss_fn(params):
        variables = {"params": params, "batch_stats": state.batch_stats}
        (logits, _), updates = state.apply_fn(
            variables,
            images,
            train=True,
            mutable=["batch_stats"],
        )
        loss = cross_entropy_loss(logits, labels)
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == labels)
        return loss, (acc, updates["batch_stats"])

    (loss, (acc, new_batch_stats)), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(batch_stats=new_batch_stats)
    return state, loss, acc


@jax.jit
def eval_step(state, images, labels):
    variables = {"params": state.params, "batch_stats": state.batch_stats}
    logits, _ = state.apply_fn(variables, images, train=False)
    loss = cross_entropy_loss(logits, labels)
    acc = jnp.mean(jnp.argmax(logits, axis=-1) == labels)
    return loss, acc


def evaluate(state, x, y, batch_size=512):
    losses, accs, counts = [], [], []
    rng = np.random.default_rng(0)
    for xb, yb in iter_minibatches(x, y, batch_size, rng=rng, train=False, augment=False):
        loss, acc = eval_step(state, jnp.asarray(xb), jnp.asarray(yb))
        n = xb.shape[0]
        losses.append(float(loss) * n)
        accs.append(float(acc) * n)
        counts.append(n)
    total = np.sum(counts)
    return {"loss": float(np.sum(losses) / total), "acc": float(np.sum(accs) / total)}


In [ ]:
# @title 6. Save / load helpers compatible with the analysis notebook

def variables_from_state(state):
    return freeze({"params": state.params, "batch_stats": state.batch_stats})


def save_tiny10_checkpoint(state, model_idx, metrics=None, history=None):
    variables = variables_from_state(state)
    ckpt_path = MODEL_SAVE_PATH / f"tiny10_model_{model_idx}.msgpack"
    with open(ckpt_path, "wb") as f:
        f.write(serialization.to_bytes(variables))

    if metrics is not None:
        with open(MODEL_SAVE_PATH / f"tiny10_model_{model_idx}_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

    if history is not None:
        pd.DataFrame(history).to_csv(MODEL_SAVE_PATH / f"tiny10_model_{model_idx}_history.csv", index=False)

    return ckpt_path


def load_tiny10_checkpoint(model_idx):
    # Mirrors the layer-matching loader: initialize a same-shaped tree, then deserialize into it.
    key = jax.random.PRNGKey(0)
    dummy_input = jnp.ones((1, 32, 32, 3), dtype=jnp.float32)
    init_vars = Tiny10(num_classes=NUM_CLASSES).init(key, dummy_input, train=False)
    path = MODEL_SAVE_PATH / f"tiny10_model_{model_idx}.msgpack"
    with open(path, "rb") as f:
        variables = serialization.from_bytes(init_vars, f.read())
    return variables


In [ ]:
# @title 7. Train one Tiny10 model

def train_one_model(model_idx, seed):
    final_path = MODEL_SAVE_PATH / f"tiny10_model_{model_idx}.msgpack"
    if SKIP_IF_EXISTS and final_path.exists():
        print(f"[model {model_idx}] found existing checkpoint, skipping: {final_path}")
        return None

    print(f"\n=== Training Tiny10 model {model_idx} / seed {seed} ===")
    state = create_state(seed=seed, num_train_examples=x_train.shape[0])
    np_rng = np.random.default_rng(seed)

    history = []
    best_test_acc = -np.inf
    best_state = None
    start_time = time.time()

    for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc=f"model {model_idx}"):
        train_losses, train_accs, counts = [], [], []
        for xb, yb in iter_minibatches(
            x_train,
            y_train,
            BATCH_SIZE,
            rng=np_rng,
            train=True,
            augment=USE_AUGMENTATION,
        ):
            state, loss, acc = train_step(state, jnp.asarray(xb), jnp.asarray(yb))
            n = xb.shape[0]
            train_losses.append(float(loss) * n)
            train_accs.append(float(acc) * n)
            counts.append(n)

        train_loss = float(np.sum(train_losses) / np.sum(counts))
        train_acc = float(np.sum(train_accs) / np.sum(counts))
        test_metrics = evaluate(state, x_test, y_test, batch_size=512)

        row = {
            "model_idx": model_idx,
            "seed": seed,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_metrics["loss"],
            "test_acc": test_metrics["acc"],
            "elapsed_min": (time.time() - start_time) / 60.0,
        }
        history.append(row)

        if test_metrics["acc"] > best_test_acc:
            best_test_acc = test_metrics["acc"]
            # JAX arrays are immutable; storing state reference is okay because state is replaced each step.
            best_state = state

        if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(
                f"model={model_idx:02d} epoch={epoch:03d} "
                f"train_acc={train_acc:.4f} test_acc={test_metrics['acc']:.4f} "
                f"best={best_test_acc:.4f}"
            )

    final_metrics = {
        "model_idx": model_idx,
        "seed": seed,
        "best_test_acc": float(best_test_acc),
        "final_test_acc": float(history[-1]["test_acc"]),
        "final_test_loss": float(history[-1]["test_loss"]),
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "base_lr": BASE_LR,
        "weight_decay": WEIGHT_DECAY,
        "normalization": "x = (image_float32 - 127.5) / 127.5",
        "checkpoint_format": "flax variables dict with params and batch_stats",
    }

    # Save the best test checkpoint as the analysis checkpoint.
    ckpt_path = save_tiny10_checkpoint(best_state, model_idx, metrics=final_metrics, history=history)
    print(f"Saved model {model_idx} to {ckpt_path}")
    print("Final metrics:", final_metrics)
    return final_metrics


In [ ]:
# @title 8. Train all 10 Tiny10 checkpoints
all_metrics = []
for model_idx, seed in enumerate(SEEDS):
    metrics = train_one_model(model_idx=model_idx, seed=seed)
    if metrics is not None:
        all_metrics.append(metrics)

# Merge all per-model metrics into one summary file.
summary_rows = []
for model_idx in range(NUM_NETWORKS):
    metrics_path = MODEL_SAVE_PATH / f"tiny10_model_{model_idx}_metrics.json"
    if metrics_path.exists():
        with open(metrics_path, "r") as f:
            summary_rows.append(json.load(f))

summary = pd.DataFrame(summary_rows).sort_values("model_idx")
summary_path = MODEL_SAVE_PATH / "tiny10_bank_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)
summary


In [ ]:
# @title 9. Reload sanity check using the exact analysis style
key = jax.random.PRNGKey(0)
dummy_input = jnp.ones((1, 32, 32, 3), dtype=jnp.float32)
init_vars = Tiny10(num_classes=NUM_CLASSES).init(key, dummy_input, train=False)
_, dummy_acts = Tiny10(num_classes=NUM_CLASSES).apply(init_vars, dummy_input, train=False)
LAYER_NAMES = sorted(list(dummy_acts.keys()), key=natural_key)
print("Layers:", LAYER_NAMES)

loaded_models = []
for i in range(NUM_NETWORKS):
    path = MODEL_SAVE_PATH / f"tiny10_model_{i}.msgpack"
    if not path.exists():
        raise FileNotFoundError(path)
    with open(path, "rb") as f:
        data = serialization.from_bytes(init_vars, f.read())
    loaded_models.append(data)

print(f"Loaded {len(loaded_models)} Tiny10 checkpoints.")

# Evaluate the reloaded checkpoints once to verify batch_stats and params deserialize correctly.
model = Tiny10(num_classes=NUM_CLASSES)
reload_rows = []
for i, variables in enumerate(loaded_models):
    # Build a state wrapper only for reuse of eval_step/evaluate.
    state = TrainState.create(
        apply_fn=model.apply,
        params=variables["params"],
        batch_stats=variables["batch_stats"],
        tx=optax.adamw(1e-3),
    )
    m = evaluate(state, x_test, y_test, batch_size=512)
    reload_rows.append({"model_idx": i, "reloaded_test_loss": m["loss"], "reloaded_test_acc": m["acc"]})

reload_df = pd.DataFrame(reload_rows)
reload_df.to_csv(MODEL_SAVE_PATH / "tiny10_reload_check.csv", index=False)
reload_df
